# Algorithms for Approximate Reducts

## Iterative Attribute Selection

---

Workshop presentation

Practical aspects of searching approximate reducts using the `skrough` package

## Agenda

- Decision tables, indiscernibility, and approximate reducts (recap)
- Greedy search strategy
- Modular pipeline architecture (hooks and stages)
- Demo: algorithm in action
- Performance profiling -- what is the bottleneck?
- GroupIndex: the core data structure
  - Three implementations, one API
  - Benchmarks
- (Bonus) Classification with reducts

## Decision Tables

A decision table is a tuple $(U, A \cup \{d\})$:

- $U$ -- universe of objects
- $A$ -- conditional attributes
- $d$ -- decision attribute

**Example: Weather**

| Outlook | Temperature | Humidity | Wind | Play |
|---------|-------------|----------|------|------|
| sunny | hot | high | weak | no |
| overcast | hot | high | weak | yes |
| rain | mild | high | weak | yes |
| ... | ... | ... | ... | ... |

$|A| = 4$, $|U| = 14$, $V_d = \{\text{yes}, \text{no}\}$

## Indiscernibility & Disorder Measures

Given $B \subseteq A$, objects $u, v \in U$ are **indiscernible** iff they have the same
values on all attributes in $B$:

$$u \, IND(B) \, v \iff \forall a \in B: a(u) = a(v)$$

Each $B$ partitions $U$ into **equivalence classes** (groups).

A **disorder measure** quantifies how impure the decision values are within groups:

- **Entropy**: $-\sum p_i \log_2 p_i$
- **Gini impurity**: $1 - \sum p_i^2$
- **Conflicts count**: number of conflicting object pairs

Goal: find $B$ that makes groups as pure as possible (low disorder).

## Approximate Reducts

Given a decision table and $\varepsilon \in [0, 1)$:

- **Base disorder**: disorder with all objects in one group (no attributes)
- **Total disorder**: disorder with all attributes $A$
- **Threshold**: $T = total + \varepsilon \cdot (base - total)$

A subset $B \subseteq A$ is an **$\varepsilon$-approximate decision reduct** iff:

$$disorder\_score(B) \leq T$$

and no proper subset of $B$ satisfies this.

Intuition: $\varepsilon = 0$ means a full reduct (max quality).
$\varepsilon > 0$ means we accept some loss for fewer attributes.

## Greedy Search Strategy

```
Input: decision table, epsilon, disorder measure
 1.  B = empty set
 2.  compute threshold T
 3.  while disorder_score(B) > T:
 4.      a* = argmax gain(B, a)        # over remaining attrs
 5.      B = B + {a*}
 6.  return B
```

- `gain(B, a) = disorder_score(B) - disorder_score(B ∪ {a})`
- Step 4 evaluates all remaining attributes per iteration
- Key challenge: efficiently compute `disorder_score(B ∪ {a})` for many candidates

## Greedy Search -- Annotated

```
 1.  B = empty set                                     <-- init hooks
 2.  compute threshold T                               <-- init hooks
 3.  while disorder_score(B) > T:                      <-- stop hooks
 4.      candidates = remaining attributes             <-- pre_candidates hooks
 5.      a* = argmax gain(B, a)                        <-- select hooks
 6.      B = B + {a*}                                  <-- inner_process hooks
 7.  return B                                          <-- prepare_result
```

Each marked point can be replaced with a different strategy -- this is the hook
architecture.

## Hook Architecture

```
ProcessingMultiStage
  ├── init hooks:      prepare data, group index, threshold
  ├── stages
  │    └── Stage
  │         ├── stop_hooks:          when to stop?
  │         ├── pre_candidates_hooks: what to consider?
  │         ├── select_hooks:         which one to pick?
  │         ├── inner_process_hooks:  what to do with it?
  │         └── finalize_hooks:       cleanup
  └── prepare_result:   format output
```

Same pipeline, different hooks = different algorithms.

Example: greedy uses approximation-threshold stop + best-gain select;
DAAR uses the same pipeline but with a stochastic acceptance test.

In [ ]:
import numpy as np
import pandas as pd

from skrough.algorithms import hooks
from skrough.algorithms.meta import processing, stage
from skrough.checks import check_if_approx_reduct
from skrough.dataprep import prepare_factorized_data
from skrough.disorder_measures import entropy
from skrough.structs.state import ProcessingState

In [ ]:
# Weather dataset
df = pd.DataFrame(
    np.array(
        [
            ["sunny", "hot", "high", "weak", "no"],
            ["sunny", "hot", "high", "strong", "no"],
            ["overcast", "hot", "high", "weak", "yes"],
            ["rain", "mild", "high", "weak", "yes"],
            ["rain", "cool", "normal", "weak", "yes"],
            ["rain", "cool", "normal", "strong", "no"],
            ["overcast", "cool", "normal", "strong", "yes"],
            ["sunny", "mild", "high", "weak", "no"],
            ["sunny", "cool", "normal", "weak", "yes"],
            ["rain", "mild", "normal", "weak", "yes"],
            ["sunny", "mild", "normal", "strong", "yes"],
            ["overcast", "mild", "high", "strong", "yes"],
            ["overcast", "hot", "normal", "weak", "yes"],
            ["rain", "mild", "high", "strong", "no"],
        ],
        dtype=object,
    ),
    columns=["Outlook", "Temperature", "Humidity", "Wind", "Play"],
)
TARGET_COLUMN = "Play"
x, x_counts, y, y_count = prepare_factorized_data(df, TARGET_COLUMN)
df

## Pipeline in Action: Setup

Build a greedy approximate reduct pipeline from hooks:

```python
grow_stage = stage.Stage.from_hooks(
    stop_hooks=[hooks.stop_hooks.stop_hook_approx_threshold],
    pre_candidates_hooks=[
        hooks.pre_candidates_hooks.pre_candidates_hook_remaining_attrs,
    ],
    select_hooks=[
        hooks.select_hooks.select_hook_attrs_disorder_score_based,
    ],
    inner_process_hooks=[
        hooks.inner_process_hooks.inner_process_hook_add_first_attr,
    ],
    inner_stop_hooks=[hooks.inner_stop_hooks.inner_stop_hook_empty],
)
```

- `stop_hook_approx_threshold`: stop when disorder_score <= threshold
- `select_hook_attrs_disorder_score_based`: pick attribute with best gain
- `inner_process_hook_add_first_attr`: add the selected attribute to result

In [ ]:
grow_stage = stage.Stage.from_hooks(
    stop_hooks=[hooks.stop_hooks.stop_hook_approx_threshold],
    init_hooks=None,
    pre_candidates_hooks=[
        hooks.pre_candidates_hooks.pre_candidates_hook_remaining_attrs,
    ],
    candidates_hooks=[hooks.process_elements.process_elements_hook_random_choice],
    select_hooks=[
        hooks.select_hooks.select_hook_attrs_disorder_score_based,
    ],
    filter_hooks=None,
    inner_init_hooks=None,
    inner_stop_hooks=[hooks.inner_stop_hooks.inner_stop_hook_empty],
    inner_process_hooks=[
        hooks.inner_process_hooks.inner_process_hook_add_first_attr,
    ],
    finalize_hooks=None,
)

get_approx_reduct = processing.ProcessingMultiStage.from_hooks(
    init_multi_stage_hooks=[
        hooks.init_hooks.init_hook_factorize_data_x_y,
        hooks.init_hooks.init_hook_single_group_index,
        hooks.init_hooks.init_hook_result_attrs_empty,
        hooks.init_hooks.init_hook_epsilon_approx_threshold,
    ],
    stages=[grow_stage],
    prepare_result_fun=hooks.prepare_result_hooks.prepare_result_hook_attrs_subset,
)

In [ ]:
# Run with epsilon = 0.4
eps = 0.4
state = ProcessingState.from_optional(processing_fun=None)
state.set_input_data_x(x)
state.set_input_data_y(y)
state.set_config_disorder_fun(entropy)
state.set_config_epsilon(eps)
state.set_config_select_attrs_disorder_score_based_max_count(1)

result = get_approx_reduct(state=state)
result

In [ ]:
# Verify the result
check_if_approx_reduct(
    x,
    x_counts,
    y,
    y_count,
    attrs=result.attrs,
    disorder_fun=entropy,
    epsilon=eps,
    check_attrs_reduction=False,
)

## Where Does the Time Go?

Profiling the full pipeline on larger data reveals:

(TODO: insert flame graph or stacked bar chart)

- GroupIndex operations dominate -- often >80% of total runtime
- `split`, `compress`, and `get_disorder_score` are the hot paths
- The greedy loop evaluates many candidates per iteration, each calling `split`

Key insight: optimizing GroupIndex is the highest-leverage change.

## GroupIndex: Core Data Structure

Represents the partition of objects into equivalence classes:

```python
@dataclass
class GroupIndex:
    index: np.ndarray    # object -> group ID
    n_groups: int        # number of distinct groups
```

**Key operations:**

| Method | What it does |
|--------|-------------|
| `split(values, values_count)` | Refine groups by a new attribute |
| `compress()` | Renumber groups to be contiguous |
| `get_disorder_score(y, y_count, fun)` | Compute disorder for current groups |
| `from_data(x, x_counts, attrs)` | Build from selected attributes |

## `split`: The Key Operation

Instead of rebuilding `GroupIndex` from scratch for each candidate:

```python
# Incremental approach:
# Start with all objects in one group
gi = GroupIndex.create_uniform(n_objects)

# For each new attribute, split existing groups
for attr in selected_attrs:
    gi = gi.split(x[:, attr], x_counts[attr])
```

`split` core formula: `new_index = old_index * values_count + values`

- Each existing group is subdivided by the new attribute's categories
- $O(n)$ time per split vs. $O(n \cdot |B|)$ for a full rebuild

In [ ]:
from skrough.structs.group_index import GroupIndex

# Demonstrate incremental GroupIndex building
gi = GroupIndex.create_uniform(len(df))
print(f"Start: {gi.n_groups} group(s)")

for attr in [0, 2]:
    gi = gi.split(x[:, attr], x_counts[attr])
    print(f"After attr {attr}: {gi.n_groups} group(s)")

gi.compress()
print(f"Compressed: {gi.n_groups} group(s)")

## Split vs Full Rebuild: Complexity

For $m$ candidates per iteration, $|B|$ selected attributes:

| Approach | Time per candidate |
|----------|-------------------|
| Full rebuild (`from_data`) | $O(n \cdot |B|)$ |
| Incremental (`split`) | $O(n)$ |

Total per iteration:
- Naive: $O(m \cdot n \cdot |B|)$ -- grows with selected attributes
- Incremental: $O(m \cdot n)$ -- independent of progress

The gap widens as the algorithm selects more attributes.

## Three Implementations of GroupIndex

Same API, different internals:

1. **Sort-based** -- sort objects by group index, then scan contiguous blocks.
   Simple, but $O(n \log n)$ per split.

2. **Hash + compress** (current) -- algebraic hashing for new index, compress to
   remove gaps, then count distributions. $O(n)$ per split.

3. **Bags / collections** (experimental) -- iterate once, maintain per-group
   collections with decision counts. $O(n)$ single-pass.

Trade-off: simplicity vs. constant factors vs. asymptotic guarantees.

## GroupIndex Benchmarks

(TODO: insert line chart -- time vs. dataset size, 3 lines)

Expected behavior:
- Sort-based: visible $O(n \log n)$ curvature
- Hash+compress: nearly linear, with compress overhead
- Bags: linear, but constant factor matters

The "best" implementation depends on the data size and characteristics.

## Full Pipeline Performance

(TODO: insert bar chart -- end-to-end timing for each GroupIndex variant)

Questions this slide answers:
- Does the GroupIndex speedup translate to real end-to-end improvement?
- How much of the pipeline time is GroupIndex vs. other operations?
- Which variant gives the best total runtime?

## Application: Classification with Reducts

Idea: use reducts as feature subsets for classification.

- Single reduct: fewer attributes, potentially lower accuracy
- **Ensemble of reducts**: combine predictions from multiple reducts via voting

Observation (not a silver bullet):
- Ensembles often match or exceed the accuracy of using all attributes
- Fewer attributes = faster prediction, smaller model
- But: requires tuning, depends on the dataset

(TODO: insert comparison chart -- all-attrs vs. single-reduct vs. ensemble)

## Summary

- **Greedy search** for approximate reducts is practical and efficient
- The **incremental split** operation on GroupIndex avoids rebuilding partitions
  from scratch, reducing complexity from $O(m \cdot n \cdot |B|)$ to $O(m \cdot n)$
- A **modular hook architecture** lets you swap strategies without changing the
  pipeline
- **Performance matters** -- profiling guides optimization, and data structure
  choices have real impact

## References

- `skrough` package: `src/skrough/`
- Theoretical foundations: `knowledgebase/`
- Example notebooks: `examples/`
- Participant notebook: (TODO: link)

## Thank You

Questions?